# Wie funktioniert ein Chatbot? – Tokenisierung, Autoregression und Kontextfenster

Chatbots wie ChatGPT wirken oft wie Zauberei: Man tippt eine Frage, und wie von selbst erscheint eine sinnvolle Antwort. In diesem Notebook nehmen wir diese Zauberei auseinander – mit echten, aber winzigen Sprachmodellen, die wir selbst laufen lassen.

**Wichtig:** Es geht hier nicht darum, ob die Antworten *gut* sind (unsere Modelle sind absichtlich sehr klein und daher nicht besonders klug). Es geht darum, *wie* die Antworten überhaupt entstehen.

Die Leitfrage für heute: *Wie kann ein Computer, der nur mit Zahlen rechnet, scheinbar sinnvolle Sätze produzieren?*

Wir schauen uns vier Bausteine an:

1. **Tokenisierung** – Wie wird aus Text eine Zahl?
2. **Autoregression** – Wie entsteht aus einem einzelnen Rateschritt ein ganzer Satz?
3. **Kontextfenster** – Woran erinnert sich das Modell, und woran nicht mehr?
4. **Chat- vs. Basismodell** – Warum "unterhält" sich das eine Modell mit uns, das andere aber nicht?

---

## Teil 1: Zwei kleine Sprachmodelle laden

Wir nutzen zwei Varianten **desselben** Modells (Qwen2.5, 0,5 Milliarden Parameter – winzig im Vergleich zu ChatGPT, aber genau deswegen schnell genug, um live im Unterricht damit zu arbeiten):

- **Basismodell** – kann nur Text fortsetzen, kennt kein "Frage-Antwort"-Format
- **Chatmodell** – dieselbe Grundarchitektur, aber zusätzlich darauf trainiert, wie ein Assistent zu antworten

Beide Modelle liegen bereits fertig heruntergeladen – lokal im Projektordner `models/`, auf bwJupyter im gemeinsamen Ordner `__shared`. Ihr müsst also nichts aus dem Internet laden.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import os

# Läuft dieses Notebook lokal (Projektordner) oder auf bwJupyter (gemeinsamer Ordner)?
modell_ordner_kandidaten = ["models", os.path.expanduser("~/work/__shared/models")]
modell_ordner = next((k for k in modell_ordner_kandidaten if os.path.isdir(k)), None)

if modell_ordner is None:
    raise FileNotFoundError(
        "Kein Modell-Ordner gefunden. Erwartet: 'models/' (lokal) oder "
        "'~/work/__shared/models' (bwJupyter). Siehe CLAUDE.md."
    )

basis_pfad = os.path.join(modell_ordner, "Qwen2.5-0.5B")
chat_pfad = os.path.join(modell_ordner, "Qwen2.5-0.5B-Instruct")

# dtype=torch.float32 erzwingen: die Gewichte liegen als bfloat16 vor, das CPU-Rechnungen
# und die Umwandlung in NumPy-Arrays (z.B. fürs Balkendiagramm) unnötig verkompliziert
tokenizer_basis = AutoTokenizer.from_pretrained(basis_pfad)
modell_basis = AutoModelForCausalLM.from_pretrained(basis_pfad, dtype=torch.float32)

tokenizer_chat = AutoTokenizer.from_pretrained(chat_pfad)
modell_chat = AutoModelForCausalLM.from_pretrained(chat_pfad, dtype=torch.float32)

print(f"Modelle geladen aus: {modell_ordner}")
print(f"Basismodell geladen: {modell_basis.num_parameters():,} Parameter")
print(f"Chatmodell geladen:  {modell_chat.num_parameters():,} Parameter")

---

## Teil 2: Tokenisierung – Wie ein Computer Text liest

Ein Computer versteht keine Buchstaben, nur Zahlen. Bevor ein Sprachmodell überhaupt mit einem Text arbeiten kann, wird dieser deshalb in **Tokens** zerlegt – kleine Text-Bausteine, oft ganze Wörter, manchmal aber auch nur Wortteile. Jeder Token bekommt eine feste Nummer aus einem festgelegten **Vokabular** (bei unserem Modell etwa 150.000 mögliche Tokens).

Schauen wir uns das an einem einfachen Satz an.

In [ ]:
satz = "Hallo, wie geht es dir?"

tokens = tokenizer_basis.tokenize(satz)
ids = tokenizer_basis.encode(satz)

print(f"Satz:          {satz}")
print(f"Tokens:        {tokens}")
print(f"Anzahl Tokens: {len(tokens)} (Anzahl Zeichen: {len(satz)})")
print(f"IDs:           {ids}")
print(f"Zurück zu Text: {tokenizer_basis.decode(ids)}")

### Ein Extremfall: sehr lange deutsche Wörter

Das Vokabular ist endlich – aber Sprache ist es nicht. Was passiert bei einem besonders langen deutschen Kompositum, das garantiert nicht als Ganzes im Vokabular steht?

In [ ]:
wort = "Donaudampfschifffahrtsgesellschaftskapitän"
tokens = tokenizer_basis.tokenize(wort)

print(f"Wort:               {wort}")
print(f"Zerlegt in {len(tokens)} Tokens: {tokens}")
print(f"\nEin einzelnes Wort kann also aus vielen Tokens bestehen -")
print(f"das Vokabular muss nur die Bruchstücke kennen, nicht jedes mögliche Wort.")

**Warum sieht `'Ã¤n'` so komisch aus?** Der Tokenizer arbeitet intern auf Ebene einzelner **Bytes**, nicht auf Ebene lesbarer Buchstaben. Das "ä" braucht als Sonderzeichen mehrere Bytes – landet so ein Byte in einem eigenen Token, sieht die rohe Darstellung kaputt aus. Sobald wir mit `.decode(...)` wieder zu einem vollständigen Text zusammensetzen (wie ganz oben), verschwindet der Effekt wieder.

---

## Teil 3: Autoregression – ein einzelner Schritt

Das Modell "kennt" die Antwort nicht als Ganzes. In jedem einzelnen Moment berechnet es nur eine **Wahrscheinlichkeitsverteilung für das allernächste Token** – basierend auf allem, was bisher da steht.

Wir geben dem Basismodell einen unvollständigen Satz und schauen uns an, welche Tokens es für am wahrscheinlichsten hält.

In [ ]:
text = "Die Hauptstadt von Frankreich ist"
eingabe = tokenizer_basis(text, return_tensors="pt")

with torch.no_grad():
    ausgabe = modell_basis(**eingabe)

logits_letztes_token = ausgabe.logits[0, -1]
wahrscheinlichkeiten = logits_letztes_token.softmax(dim=-1)

top5 = torch.topk(wahrscheinlichkeiten, 5)

print(f"Text: '{text}'")
print("Die 5 wahrscheinlichsten nächsten Tokens:\n")
for wert, idx in zip(top5.values, top5.indices):
    token_text = tokenizer_basis.decode(idx)
    print(f"   {token_text!r:15s} {wert.item():.1%}")

Dasselbe noch einmal als Balkendiagramm – so wird deutlich, dass das Modell nicht "weiß", sondern nur **schätzt**, wie wahrscheinlich jeder Kandidat ist.

In [ ]:
import matplotlib.pyplot as plt

woerter = [tokenizer_basis.decode(idx).strip() for idx in top5.indices]
werte = top5.values.detach().numpy()

plt.bar(woerter, werte, color="steelblue")
plt.title(f"Wahrscheinlichkeit für das nächste Token nach:\n'{text}'")
plt.ylabel("Wahrscheinlichkeit")
plt.tight_layout()
plt.show()

---

## Teil 4: Autoregression – viele Schritte hintereinander

Ein ganzer Chatbot-Text entsteht, indem Teil 3 immer wieder wiederholt wird: nächstes Token vorhersagen, anhängen, mit dem jetzt längeren Text von vorne beginnen. Das eigene Ergebnis wird also zur eigenen neuen Eingabe – daher der Name **Auto**-Regression.

Wir schreiben die Schleife diesmal selbst (statt der fertigen `generate()`-Funktion), damit der Mechanismus sichtbar bleibt.

**Aber wann hört diese Schleife eigentlich auf?** Im Vokabular gibt es einen speziellen Token, den **End-of-Sequence-Token (EOS)**. Technisch ist er ein Token wie jeder andere – er bekommt bei jedem Schritt genau wie alle anderen eine Wahrscheinlichkeit zugewiesen. Sagt das Modell diesen Token als wahrscheinlichstes nächstes Token voraus, ist das die Konvention für "Ich bin fertig" – und unsere Schleife sollte an dieser Stelle abbrechen, statt stur weiterzumachen.

In [ ]:
text = "Es war einmal ein Junge, der"

for schritt in range(1, 71):
    eingabe = tokenizer_basis(text, return_tensors="pt")
    with torch.no_grad():
        logits = modell_basis(**eingabe).logits[0, -1]
    naechste_id = logits.argmax()

    if naechste_id.item() == tokenizer_basis.eos_token_id:
        print(f"EOS-Token erkannt nach Schritt {schritt} - Modell stoppt von selbst.")
        break

    text += tokenizer_basis.decode(naechste_id)
    if schritt % 10 == 0:
        print(f"Nach Schritt {schritt:2d}: {text}")
else:
    print(f"Kein EOS nach 70 Schritten - die Schleife wurde nur durch unser Limit gestoppt.")

print(f"\nEndgültiger Text:\n{text}")

**Was fällt dir auf?** Erstens: Der Text wiederholt sich nach einer Weile oder es schleichen sich Grammatikfehler ein – winzige Ungenauigkeiten addieren sich von Schritt zu Schritt auf. Zweitens, und das ist der eigentliche Punkt hier: Der EOS-Token kam nie – unser Basismodell wurde beim Training nie darauf trainiert, mitten in einer Geschichte zu erkennen "jetzt ist Schluss". Ein größeres, besser trainiertes Modell driftet nicht so schnell ab, aber auch dort entscheidet exakt derselbe Mechanismus (EOS ja/nein), wann eine Antwort endet.

---

## Teil 5: Kontextfenster – was das Modell noch "sieht"

Ein Sprachmodell schaut bei jedem Vorhersageschritt nicht auf den *gesamten* bisherigen Text, sondern nur auf die letzten **N Tokens** – das nennt man das **Kontextfenster**. Alles, was davor liegt, ist für das Modell schlicht nicht mehr sichtbar. Selbst wenn dort etwas Wichtiges stand, zum Beispiel dein Name.

Das echte Kontextfenster unseres Modells ist mit tausenden Tokens viel zu groß, um den Effekt schnell zu zeigen. Deshalb bauen wir uns ein **eigenes, künstlich verkleinertes Kontextfenster**: eine Funktion, die vor der Vorhersage einfach alle Tokens abschneidet, die weiter als `fenstergroesse` Tokens zurückliegen.

Wir nutzen dafür das **Basismodell** und lassen es den Satz einfach fortsetzen ("Ich heiße ___") – so sehen wir unverfälscht, ob der Name noch im sichtbaren Bereich steckt oder nicht.

In [ ]:
def vervollstaendige_mit_fenster(text, fenstergroesse):
    eingabe_ids = tokenizer_basis(text, return_tensors="pt").input_ids
    sichtbare_ids = eingabe_ids[:, -fenstergroesse:]  # nur die letzten n Tokens "sieht" das Modell

    ausgabe = modell_basis.generate(
        sichtbare_ids, max_new_tokens=10, pad_token_id=tokenizer_basis.eos_token_id
    )
    fortsetzung = tokenizer_basis.decode(ausgabe[0, sichtbare_ids.shape[1]:], skip_special_tokens=True)
    return fortsetzung


fuelltext = "Die Sonne schien und die Vögel zwitscherten. " * 40
text_mit_name = f"Mein Name ist Alex. {fuelltext}Ich heiße"

print(f"Gesamtlänge des Texts: {len(tokenizer_basis(text_mit_name).input_ids)} Tokens\n")

for fenster in [1000, 100, 30, 15]:
    fortsetzung = vervollstaendige_mit_fenster(text_mit_name, fenster)
    print(f"Fenstergröße {fenster:4d} Tokens -> 'Ich heiße{fortsetzung}'")

Bei großem Fenster liegt "Mein Name ist Alex" noch innerhalb der sichtbaren Tokens – das Modell vervollständigt korrekt mit "Alex". Wird das Fenster klein genug, fällt der Name aus dem sichtbaren Bereich heraus – das Modell greift dann auf irgendetwas anderes zurück, das noch sichtbar ist (z.B. Wörter aus dem Fülltext), weil es den Namen schlicht nicht mehr "sehen" kann.

---

## Teil 6: Chat- vs. Basismodell

Beide Modelle haben dieselbe Architektur und dasselbe Vokabular. Der Unterschied entsteht durch zusätzliches Training: Das Chatmodell wurde darauf trainiert, Fragen zu beantworten statt Text nur fortzusetzen.

Testen wir dieselbe Frage an beiden Modellen.

In [ ]:
frage = "Was ist die Hauptstadt von Frankreich?"

eingabe = tokenizer_basis(frage, return_tensors="pt")
ausgabe = modell_basis.generate(
    **eingabe, max_new_tokens=30, pad_token_id=tokenizer_basis.eos_token_id
)

print("Basismodell bekommt die Frage roh als Text:\n")
print(tokenizer_basis.decode(ausgabe[0], skip_special_tokens=True))

Das Basismodell setzt den Text meist einfach fort (z.B. mit weiteren Fragen oder einer Auflistung), statt gezielt zu antworten – es kennt schließlich kein "Frage-Antwort"-Format.

Beim Chatmodell nutzen wir jetzt ein sogenanntes **Chat-Template**. Das ist keine neue Zauberei, sondern eine feste Textkonvention mit Spezial-Tokens, die dem Modell markiert: "Hier beginnt eine Nutzer-Nachricht, hier soll eine Antwort folgen." Schauen wir uns den rohen, formatierten Text einmal direkt an, bevor wir ihn ans Modell schicken.

In [ ]:
nachrichten = [{"role": "user", "content": frage}]
formatiert = tokenizer_chat.apply_chat_template(
    nachrichten, tokenize=False, add_generation_prompt=True
)

print("So sieht der Text wirklich aus, den das Chat-Modell bekommt:\n")
print(formatiert)

In [ ]:
eingabe_chat = tokenizer_chat(formatiert, return_tensors="pt")
ausgabe_chat = modell_chat.generate(
    **eingabe_chat, max_new_tokens=30, pad_token_id=tokenizer_chat.eos_token_id
)
antwort_chat = tokenizer_chat.decode(
    ausgabe_chat[0, eingabe_chat.input_ids.shape[1]:], skip_special_tokens=True
)

print("Chat-Modell antwortet:\n")
print(antwort_chat)

### Kennt das Basismodell das Chat-Format?

Der Tokenizer des Basismodells kennt das Chat-Template tatsächlich auch – es gehört zur ganzen Modellfamilie, nicht nur zur Chat-Variante. Die eigentliche Frage ist also nicht "kennt der Tokenizer das Format", sondern: **weiß das Basismodell auch, wie es damit umgehen soll?**

In [ ]:
formatiert_basis = tokenizer_basis.apply_chat_template(
    nachrichten, tokenize=False, add_generation_prompt=True
)
eingabe_basis = tokenizer_basis(formatiert_basis, return_tensors="pt")
ausgabe_basis = modell_basis.generate(
    **eingabe_basis, max_new_tokens=30, pad_token_id=tokenizer_basis.eos_token_id
)
antwort_basis = tokenizer_basis.decode(
    ausgabe_basis[0, eingabe_basis.input_ids.shape[1]:], skip_special_tokens=True
)

print("Basismodell antwortet auf dasselbe Chat-Format:\n")
print(antwort_basis)

Schauen wir uns das nicht nur am Endergebnis an, sondern direkt am Mechanismus: Wir zählen mit, nach wie vielen Schritten (falls überhaupt) der **EOS-Token** aus Teil 4 auftaucht – für beide Modelle, mit demselben Chat-formatierten Prompt.

In [ ]:
def generiere_bis_eos(tokenizer, modell, formatierter_text, max_schritte=30):
    ids = tokenizer(formatierter_text, return_tensors="pt").input_ids
    start_laenge = ids.shape[1]

    for schritt in range(1, max_schritte + 1):
        with torch.no_grad():
            logits = modell(ids).logits[0, -1]
        naechste_id = logits.argmax()

        if naechste_id.item() == tokenizer.eos_token_id:
            print(f"  EOS-Token erkannt nach Schritt {schritt} - Modell stoppt von selbst.")
            break
        ids = torch.cat([ids, naechste_id.view(1, 1)], dim=1)
    else:
        print(f"  Kein EOS nach {max_schritte} Schritten.")

    return tokenizer.decode(ids[0, start_laenge:], skip_special_tokens=True)


print("Chat-Modell:")
print(f"  Antwort: {generiere_bis_eos(tokenizer_chat, modell_chat, formatiert)!r}\n")

print("Basismodell (mit demselben Chat-Format):")
print(f"  Antwort: {generiere_bis_eos(tokenizer_basis, modell_basis, formatiert_basis)!r}")

Genau das bestätigt sich: Das Chatmodell sagt nach wenigen Schritten selbst den EOS-Token voraus und stoppt sauber nach "Paris." – das Basismodell hält EOS hier nie für das wahrscheinlichste nächste Token und rutscht stattdessen in unzusammenhängenden Text ab.

Das zeigt den eigentlichen Unterschied zwischen Chat- und Basismodell: **Das Format allein reicht nicht.** Beide Modelle kennen denselben EOS-Token im Vokabular und dasselbe Chat-Template – aber nur das Chatmodell wurde zusätzlich trainiert, EOS nach einer vollständigen Antwort auch zuverlässig vorherzusagen.

---

## Zusammenfassung & Ausblick

Das haben wir uns heute angeschaut:

*   **Tokenisierung** – Text wird in Zahlen (Tokens) zerlegt, bevor ein Modell überhaupt etwas verarbeiten kann
*   **Autoregression** – das Modell sagt immer nur das *nächste* Token voraus; ein ganzer Text entsteht erst durch Wiederholung dieses einen Schritts
*   **EOS-Token** – ein ganz normaler Token im Vokabular, der signalisiert "fertig"; nur trainierte Chatmodelle sagen ihn zuverlässig an der richtigen Stelle voraus
*   **Kontextfenster** – das Modell sieht nur eine begrenzte Anzahl der letzten Tokens; alles davor ist schlicht nicht mehr sichtbar
*   **Chat- vs. Basismodell** – "Chat"-Fähigkeit ist keine andere Architektur, sondern zusätzliches Training plus eine Formatierungskonvention (Chat-Template) mit Spezial-Tokens

### Was kommt als nächstes?

Echte Chatbots wie ChatGPT nutzen dieselben Bausteine – nur mit viel größeren Modellen (hunderte Milliarden statt 0,5 Milliarden Parameter) und zusätzlichem Training, damit die Antworten hilfreich und sicher sind. Der Mechanismus im Kern bleibt aber genau derselbe, den wir hier gerade selbst nachgebaut haben.

---

## Bonus: Quatsch ein bisschen mit dem Chatbot

Jetzt weißt du, wie das Chat-Modell "denkt" – Zeit, es einfach mal auszuprobieren. Der Code unten baut eine kleine Chat-Schleife: Du tippst eine Nachricht, das Modell antwortet, deine Nachricht und die Antwort werden dem Gesprächsverlauf hinzugefügt, und die nächste Runde bekommt den ganzen bisherigen Verlauf zu sehen (bis das Kontextfenster – siehe Teil 5 – irgendwann voll ist).

Diesmal nutzen wir wieder unsere eigene Schleife aus Teil 4 statt `generate()` – dadurch siehst du die Antwort **Token für Token live entstehen**, genau wie in Teil 3 und 4 besprochen, inklusive des EOS-Stopps aus Teil 4.

Tipp zum Ausprobieren: Frag nach deinem Namen, nachdem du ihn ein paar Nachrichten vorher genannt hast – erinnert sich das Modell? Oder stell absichtlich eine Fangfrage. Das Modell ist winzig (0,5 Milliarden Parameter) – erwarte ruhig auch mal Unsinn.

Zum Beenden `ende` eingeben.

In [ ]:
verlauf = [
    {"role": "system", "content": "Du bist ein hilfsbereiter Chatbot für Schülerinnen und Schüler."}
]

print("Chatte mit dem Modell! ('ende' zum Beenden)\n")

while True:
    nutzereingabe = input("Du: ")
    if nutzereingabe.strip().lower() == "ende":
        break

    verlauf.append({"role": "user", "content": nutzereingabe})
    formatiert_verlauf = tokenizer_chat.apply_chat_template(
        verlauf, tokenize=False, add_generation_prompt=True
    )
    ids = tokenizer_chat(formatiert_verlauf, return_tensors="pt").input_ids

    # Token für Token selbst erzeugen (wie in Teil 4), statt generate() zu nutzen -
    # so siehst du die Antwort live entstehen, inklusive EOS-Stopp.
    print("Bot: ", end="", flush=True)
    neue_token_ids = []
    for _ in range(150):
        with torch.no_grad():
            logits = modell_chat(ids).logits[0, -1]
        naechste_id = logits.argmax()
        if naechste_id.item() == tokenizer_chat.eos_token_id:
            break
        neue_token_ids.append(naechste_id.item())
        print(tokenizer_chat.decode(naechste_id), end="", flush=True)
        ids = torch.cat([ids, naechste_id.view(1, 1)], dim=1)
    print("\n")

    antwort = tokenizer_chat.decode(neue_token_ids)
    verlauf.append({"role": "assistant", "content": antwort})

---

# Übungsaufgaben – Chatbot-Mechanik selbst ausprobieren

Jetzt bist du dran! In den folgenden Aufgaben nutzt du `tokenizer_basis`, `tokenizer_chat`, `modell_basis` und `modell_chat` von oben weiter.

### Aufgabe 1 (leicht): Tokenanzahl vergleichen

Nicht jedes Wort wird gleich stark zerlegt. Kurze, häufige englische Wörter brauchen oft nur einen Token, lange deutsche Wörter deutlich mehr.

**Aufgabe:**

1. Erstelle eine Liste `woerter` mit mindestens 5 Wörtern deiner Wahl – mische kurze deutsche Wörter, lange deutsche Komposita und ein paar englische Wörter.
2. Berechne für jedes Wort die Anzahl der Tokens mit `tokenizer_basis.tokenize(wort)`.
3. Gib für jedes Wort Zeichenanzahl und Tokenanzahl aus.
4. Welches Wort hat das schlechteste Verhältnis (Tokens pro Zeichen)?

*Tipp:* `len(wort)` gibt die Zeichenanzahl, `len(tokenizer_basis.tokenize(wort))` die Tokenanzahl.

In [ ]:
# Aufgabe 1: Tokenanzahl vergleichen

woerter = ["Haus", "Grundstücksverkehrsgenehmigungszuständigkeitsübertragung", "computer"]

for wort in woerter:
    # anzahl_tokens = ...
    # print(f"{wort:<55} {len(wort):>4} Zeichen, {anzahl_tokens:>3} Tokens")
    pass

### Aufgabe 2 (mittel): Die Kontextfenster-Schwelle finden

In Teil 5 haben wir nur 4 feste Fenstergrößen getestet. Jetzt willst du die **genaue Schwelle** finden: Ab welcher Fenstergröße "vergisst" das Modell den Namen?

**Aufgabe:**

1. Nutze die Funktion `vervollstaendige_mit_fenster` und den Text `text_mit_name` von oben.
2. Teste systematisch mehrere Fenstergrößen zwischen 10 und 100 (z.B. in 10er-Schritten mit einer Schleife).
3. Gib für jede Fenstergröße die Fortsetzung aus.
4. Bestimme die kleinste Fenstergröße, bei der "Alex" noch in der Fortsetzung vorkommt (Tipp: `"alex" in fortsetzung.lower()`).

*Tipp:* `range(10, 101, 10)` erzeugt die Werte 10, 20, 30, ..., 100.

In [ ]:
# Aufgabe 2: Kontextfenster-Schwelle finden

kleinste_erinnerte_groesse = None

for fenster in range(10, 101, 10):
    # fortsetzung = vervollstaendige_mit_fenster(text_mit_name, fenster)
    # print(f"Fenstergröße {fenster:4d} -> 'Ich heiße{fortsetzung}'")
    # if "alex" in fortsetzung.lower():
    #     kleinste_erinnerte_groesse = fenster
    pass

# print(f"\nKleinste Fenstergröße mit Erinnerung: {kleinste_erinnerte_groesse}")

### Aufgabe 3 (schwer): Eigene Mehrfach-Turn-Konversation

Bisher hatte unsere `nachrichten`-Liste nur eine einzige Nutzer-Nachricht. Echte Chat-Konversationen bestehen aber aus mehreren Runden (Nutzer fragt, Modell antwortet, Nutzer fragt weiter) und können sogar eine **System-Nachricht** enthalten, die dem Modell eine Rolle vorgibt.

**Aufgabe:**

1. Baue eine `eigene_nachrichten`-Liste mit mindestens vier Einträgen: einer `"system"`-Nachricht (z.B. "Du bist ein hilfsbereiter Assistent, der immer auf Deutsch antwortet"), einer ersten `"user"`-Frage, einer erfundenen `"assistant"`-Antwort darauf, und einer zweiten `"user"`-Nachfrage.
2. Wandle sie mit `tokenizer_chat.apply_chat_template(...)` in Text um und schau dir das Ergebnis an – wie viele Rollen-Markierungen siehst du?
3. Lass `modell_chat` darauf antworten.
4. Vergleiche: Reagiert das Modell auf die *zweite* Frage so, als würde es sich an die erste Runde erinnern?

*Tipp:* Jede Nachricht ist ein Dictionary: `{"role": "system", "content": "..."}`.

In [ ]:
# Aufgabe 3: Eigene Mehrfach-Turn-Konversation

eigene_nachrichten = [
    # {"role": "system", "content": "..."},
    # {"role": "user", "content": "..."},
    # {"role": "assistant", "content": "..."},
    # {"role": "user", "content": "..."},
]

# eigen_formatiert = tokenizer_chat.apply_chat_template(eigene_nachrichten, tokenize=False, add_generation_prompt=True)
# print(eigen_formatiert)

# eigene_eingabe = tokenizer_chat(eigen_formatiert, return_tensors="pt")
# eigene_ausgabe = modell_chat.generate(**eigene_eingabe, max_new_tokens=40, pad_token_id=tokenizer_chat.eos_token_id)
# eigene_antwort = tokenizer_chat.decode(eigene_ausgabe[0, eigene_eingabe.input_ids.shape[1]:], skip_special_tokens=True)
# print(f"\nAntwort: {eigene_antwort}")

---

## Geschafft!

Du hast einem Chatbot beim Denken zugeschaut und dabei gelernt:

- Wie **Tokenisierung** Text in verarbeitbare Zahlen zerlegt
- Wie **Autoregression** aus einem einzelnen Vorhersageschritt einen ganzen Text macht
- Warum das **Kontextfenster** bestimmt, woran sich ein Modell noch "erinnert"
- Warum ein **Chatmodell** anders reagiert als ein **Basismodell** – nicht wegen einer anderen Architektur, sondern wegen zusätzlichem Training und einem Chat-Template

Die wichtigste Erkenntnis: Ein Chatbot ist im Kern ein sehr einfacher Mechanismus, der Schritt für Schritt das nächste Token schätzt – die scheinbare "Intelligenz" entsteht erst dadurch, dass dieser einfache Mechanismus millionenfach trainiert und wiederholt wird.